#Merton Model Implementation
The goal of my project is to assess a bond portfolio's risk. By inputting some key financial information about each company in our portfolio into a Merton Model, we can find an estimation of the probability of any given company dissolving. This is key because if a company dissolves our bond with the company will default and money will be lost. This is the core of the project and I will detail the formulas used in the model here.

In [2]:
import yfinance as yf
import numpy as np
import pandas as probability_default
from scipy.stats import norm
from scipy.optimize import root

N = norm.cdf

We import libraries needed, and load a cumulative normal distribution

Our inputs in order will be:

1.  **Market Capitalization (Market Cap)**
    * Current Stock Price * Shares Outstanding

2.  **Default Barrier**
    * The specific value of a company's assets at which it can no longer meet its financial obligations (therefore defaults). An estimate of this is could be the company's total liabilities.

3.  **Equity Volatility**
    * How volatile a stock's price is

4.  **Risk-Free Interest Rate**
    * Risk free interest rate - the rate of return on a risk free investment, like the US treasury bonds (even these aren't 100% safe these days, though!).

5.  **Time to Maturity**
    * The remaining time of the bond in years. Longer time frame = more risk



In [3]:
def merton_model(E, D, equity_volatility, r, T):
    # Defining the system of equations to solve
    def equations(x):
        V, sigma_v = x
        d1 = (np.log(V/D) + (r + 0.5 * sigma_v**2) * T) / (sigma_v * np.sqrt(T))
        d2 = d1 - sigma_v * np.sqrt(T)
        eq1 = V * N(d1) - D * np.exp(-r * T) * N(d2) - E
        eq2 = N(d1) * V * sigma_v / E - equity_volatility # Equation for equity volatility
        return [eq1, eq2]

    # Initial guess: V = E + D, sigma_v = equity_volatility * E / (E + D)
    V_guess = E + D
    sigma_v_guess = equity_volatility * E / (E + D)
    sol = root(equations, [V_guess, sigma_v_guess])
    V, sigma_v = sol.x

    distance_default = (np.log(V/D) + (r - 0.5 * sigma_v**2) * T) / (sigma_v * np.sqrt(T))
    probability_default = N(-distance_default)
    return float(V), float(sigma_v), float(distance_default), float(probability_default)

### Merton Model Equations

The core of the Merton model involves solving these two equations simultaneously for Asset Value and Asset Volatility:

#### 1. Equity Value Equation (Black-Scholes Call Option Formula)
$$
E = V \cdot N(d_1) - D \cdot e^{-rT} \cdot N(d_2)
$$

#### 2. Equity Volatility Equation (Derived from Ito's Lemma)
$$
\sigma_e = \frac{N(d_1) \cdot V \cdot \sigma_v}{E}
$$

- $d_1 = \frac{\ln(V/D) + (r + 0.5\sigma_v^2)T}{\sigma_v\sqrt{T}}$
- $d_2 = d_1 - \sigma_v\sqrt{T}$
- $N(\cdot)$ = Cumulative distribution function of the standard normal distribution
- $E$ = Market value of equity (Market Cap)
- $D$ = Default barrier (Total Liabilities)
- $\sigma_e$ = Equity volatility
- $r$ = Risk-free interest rate
- $T$ = Time to maturity
- $V$ = Value of company's assets (to be found)
- $\sigma_v$ = Volatility of company's assets (to be found)

With our newfound value estimate for a company's assets and the volatility. We can plug these into a formula which works out a "Distance to Default": how close a company is to hitting net zero.

#### Distance to Default
$$
\text{Distance to Default} = \frac{\ln(V/D) + (r - 0.5\sigma_v^2)T}{\sigma_v\sqrt{T}}
$$


# Results: APPL
Inputting the financials of Apple, we find a probability of 0.0000055677 in the next five years. The reason for this being so low is because the company has an extremely strong balance sheet at a very large scale. On paper, it is near enough impossible for them to face bankruptcy; in reality it's a little more likely than the value we produce due to other qualatative factors or risk, such Operational or reputational risk.

In [4]:
print(merton_model(3445050426286.0796,308030000000.0, 0.27472650331239595,0.0383,5))

(3699396691212.005, 0.2558374483867807, 4.393861195635436, 5.5677465238764525e-06)


# SNAP
Inputting the financials of a slightly smaller company as Snapchat provides more interesting results. At the time of writing the company is not performing as well as it had in previous years, leading to reduced stock price as well. Our model predicts a much greater probability of 0.347621 for defaulting in the next 5 years. This is likely a little bit of an overestimate because firms struggling often have other options for a breath of new life, like refinancing or M&A.

In [5]:
print(merton_model(9971228304,5485587000, 0.6668512868005332,0.0383,5))

(13773814747.801476, 0.5144062688733219, 0.3917510986009366, 0.34762106518808794)


# Conclusion
The merton model is a very strong abstract tool to generate clear cut probabilities of a company going bust. It loses its accuracy when we look at the healthiest companies and the ones in distress. We will consider this when making our bond portfolio.